# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [ ]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 887 bytes | 221.00 KiB/s, done.
From https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2
   9576992..5d9b0bf  main       -> origin/main
Updating 9576992..5d9b0bf
Fast-forward
 notebooks/level1_classic_cnns.ipynb | 41 +++++++++++++++++++++++--------------
 1 file changed, 26 insertions(+), 15 deletions(-)
/content/2026-HYU-AUE8088-PA2


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [7]:
import wandb; wandb.login()   # API key 입력

True

In [8]:
WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [9]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋이 이미 존재합니다 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [10]:
from torch import nn

def train_one(model_fn, name, epochs=20):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"../checkpoints/level1_{name}.pth")
    return model, history

# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=4.1137  val_avg_MF1=0.3393  per={'weather': 0.125, 'scene': 0.2531017369727047, 'timeofday': 0.6397102722674687}


[epoch 02/30] train_loss=2.5311  val_avg_MF1=0.3396  per={'weather': 0.125, 'scene': 0.2525879917184265, 'timeofday': 0.6411762801154993}


[epoch 03/30] train_loss=2.3278  val_avg_MF1=0.3760  per={'weather': 0.18671932802367586, 'scene': 0.30156145018678276, 'timeofday': 0.6395997929187206}


[epoch 04/30] train_loss=2.2513  val_avg_MF1=0.4022  per={'weather': 0.19623233908948193, 'scene': 0.3707133058984911, 'timeofday': 0.6397813136943572}


...

[epoch 27/30] train_loss=1.0604  val_avg_MF1=0.6326  per={'weather': 0.5359778246542953, 'scene': 0.5609435392754357, 'timeofday': 0.8008353898110113}


[epoch 28/30] train_loss=1.0216  val_avg_MF1=0.6257  per={'weather': 0.533053165915637, 'scene': 0.5322207702337507, 'timeofday': 0.811852031655449}


[epoch 29/30] train_loss=1.0222  val_avg_MF1=0.6341  per={'weather': 0.5283099759843947, 'scene': 0.5620292561469032, 'timeofday': 0.811841403290428}


[epoch 30/30] train_loss=0.9852  val_avg_MF1=0.6377  per={'weather': 0.5334524826515268, 'scene': 0.5715429054087874, 'timeofday': 0.8080713536452698}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val/avg_macro_f1,▃▃▃▅▁▄▅▂▅▆▆▅▆▆▆▅▆▆▇▆▇▇▇█▇▇████
val/mf1_scene,▃▃▁▃▂▂▃▂▃▄▃▅▃▅▃▂▅▃▆▄▆▆▆▇▇▇█▇██
val/mf1_timeofday,▅▅▆▇▁▆▇▁▇██▇█▇█▇██▇█████▇▇████
val/mf1_weather,▁▁▂▂▁▃▄▅▅▅▄▄▅▅▆▅▅▅█▆▇▇▇▇▇▇████
epoch,30
lr,0
train/loss,0.98522
val/avg_macro_f1,0.63769


In [ ]:
# Uncertainty Weighting 실험 (Kendall et al., CVPR 2018)
# Reference: https://arxiv.org/abs/1705.07115
from src.losses.imbalanced import UncertaintyWeighting

def train_one_uw(model_fn, name, epochs=30):
    """Uncertainty Weighting 으로 학습. σ 값이 wandb 에 자동 로깅된다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    uw = UncertaintyWeighting(list(ATTRIBUTES)).to(device)

    optim = torch.optim.AdamW(
        list(model.parameters()) + list(uw.parameters()),
        lr=3e-4, weight_decay=5e-4,
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}-uw",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weighting": "uncertainty",
        },
        tags=WANDB_TAGS + [name, "uncertainty-weighting"],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger, uw=uw)
    history = trainer.fit(train_loader, val_loader)

    print(f"\n[{name}] 최종 σ 값 (클수록 어려운 task):")
    for attr, sigma in uw.sigmas().items():
        print(f"  σ_{attr:<10s} = {sigma:.4f}  (가중치 ≈ {1 / (2 * sigma**2):.4f})")

    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"../checkpoints/level1_{name}_uw.pth")
    return model, history, uw

vgg_uw,  vgg_hist_uw,  vgg_uw_module  = train_one_uw(VGG16,    "vgg16",    epochs=30)
r18_uw,  r18_hist_uw,  r18_uw_module  = train_one_uw(resnet18, "resnet18", epochs=30)
r50_uw,  r50_hist_uw,  r50_uw_module  = train_one_uw(resnet50, "resnet50", epochs=30)

## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.